# 01 — EDA: Maintenance Work Order Corpus

Goals:
- Understand the synthetic corpus structure
- Text length distribution, vocabulary richness per category
- Class balance across 6 failure categories
- Most frequent terms per category (word clouds or bar charts)
- Identify any data quality issues before modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys; sys.path.insert(0, '..')
from data.synthetic_generator import generate_corpus, main as gen_main
from src.nlp_pipeline import preprocess_series

%matplotlib inline
sns.set_theme(style='darkgrid')

In [ ]:
# Generate corpus if not already on disk
import os
if not Path('../data/work_orders.csv').exists():
    os.chdir('..')
    gen_main()
    os.chdir('notebooks')

df = pd.read_csv('../data/work_orders.csv')
print(df.shape)
df.head(3)

In [ ]:
# Class distribution
print(df.failure_category.value_counts())
df.failure_category.value_counts().plot(kind='barh', figsize=(7,4))
plt.title('Work Order Class Distribution')
plt.tight_layout(); plt.show()

In [ ]:
# Text length distribution per category
df['text_len'] = df.text.str.split().str.len()
df.boxplot(column='text_len', by='failure_category', figsize=(10,5), rot=30)
plt.suptitle('Text Length (words) by Category')
plt.tight_layout(); plt.show()
print(df.groupby('failure_category').text_len.describe().round(1))

In [ ]:
# Vocabulary richness per category
from collections import Counter
for cat in df.failure_category.unique():
    sub = df[df.failure_category==cat]
    words = ' '.join(sub.text).lower().split()
    uniq = len(set(words))
    print(f'{cat:35s}: {len(sub):,} docs | {len(words):,} tokens | {uniq:,} unique')

In [ ]:
# Top 15 terms per category (after preprocessing)
from sklearn.feature_extraction.text import CountVectorizer
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, cat in zip(axes.flatten(), df.failure_category.unique()):
    sub = df[df.failure_category==cat].text.tolist()
    cv = CountVectorizer(max_features=15, stop_words='english', ngram_range=(1,2))
    X = cv.fit_transform(sub)
    terms = cv.get_feature_names_out()
    counts = X.toarray().sum(axis=0)
    ax.barh(terms, counts)
    ax.set_title(cat.replace('_',' '))
plt.suptitle('Top Bigrams per Category')
plt.tight_layout(); plt.show()